In [5]:
"""
Ensemble Model Comparison
=========================
Ensemble A: LSTM + LightGBM
Ensemble B: TCN + TFT + XGBoost

Strategies per group:
  - Simple Average
  - Sharpe-weighted Average

Winner selected by: Directional Accuracy
"""

import numpy as np
import pandas as pd

# ─────────────────────────────────────────────
# 0.  CONFIG
# ─────────────────────────────────────────────
DATA_PATH = "all_models_predictions.csv"   

GROUP_A = ["LSTM", "LightGBM"]
GROUP_B = ["TCN", "TFT", "XGBoost"]

# ─────────────────────────────────────────────
# 1.  METRICS
# ─────────────────────────────────────────────

def sharpe(returns: np.ndarray, annualize: int = 252) -> float:
    if returns.std() == 0:
        return 0.0
    return (returns.mean() / returns.std()) * np.sqrt(annualize)


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def directional_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.sign(y_true) == np.sign(y_pred))


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    signal_returns = np.sign(y_pred) * y_true
    return {
        "DirAcc":  directional_accuracy(y_true, y_pred),
        "Sharpe":  sharpe(signal_returns),
        "RMSE":    rmse(y_true, y_pred),
        "MeanRet": signal_returns.mean(),
        "RetStd":  signal_returns.std(),
    }


# ─────────────────────────────────────────────
# 2.  DATA LOADING & PIVOT
# ─────────────────────────────────────────────

def load_and_pivot(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"])

    pivot = df.pivot_table(
        index=["date", "fold_id"],
        columns="model",
        values="y_pred",
        aggfunc="mean",
    ).reset_index()
    pivot.columns.name = None

    y_true_map = df.groupby(["date", "fold_id"])["y_true"].first().reset_index()
    pivot = pivot.merge(y_true_map, on=["date", "fold_id"])
    pivot = pivot.sort_values(["fold_id", "date"]).reset_index(drop=True)
    return pivot


# ─────────────────────────────────────────────
# 3.  ENSEMBLE STRATEGIES
# ─────────────────────────────────────────────

def simple_avg(pivot: pd.DataFrame, models: list) -> np.ndarray:
    return pivot[models].mean(axis=1).values


def sharpe_weighted_avg(pivot: pd.DataFrame, models: list) -> np.ndarray:
    weights = {}
    for m in models:
        sig_ret = np.sign(pivot[m].values) * pivot["y_true"].values
        weights[m] = max(sharpe(sig_ret), 0)   # floor negative Sharpe at 0

    w = np.array([weights[m] for m in models], dtype=float)
    if w.sum() == 0:
        w = np.ones(len(models))
    w /= w.sum()

    print(f"Sharpe weights: { {m: round(w[i], 3) for i, m in enumerate(models)} }")
    return (pivot[models].values * w).sum(axis=1)


# ─────────────────────────────────────────────
# 4.  EVALUATE ONE GROUP
# ─────────────────────────────────────────────

def evaluate_group(pivot: pd.DataFrame, models: list, group_name: str) -> dict:
    print(f"\n{'='*60}")
    print(f"  GROUP {group_name}:  {models}")
    print(f"{'='*60}")

    y_true = pivot["y_true"].values

    # Individual model metrics
    print("\n── Individual model metrics ──")
    for m in models:
        met = compute_metrics(y_true, pivot[m].values)
        print(f"  {m:15s}  DirAcc={met['DirAcc']:.4f}  Sharpe={met['Sharpe']:+.3f}  RMSE={met['RMSE']:.5f}")

    # Ensemble predictions
    print()
    strategies = {
        "simple_avg":      simple_avg(pivot, models),
        "sharpe_weighted": sharpe_weighted_avg(pivot, models),
    }

    results = {name: compute_metrics(y_true, preds) for name, preds in strategies.items()}
    metrics_df = pd.DataFrame(results).T
    metrics_df.index.name = "Strategy"
    metrics_df = metrics_df.sort_values("DirAcc", ascending=False)

    print(f"\n── Ensemble strategy comparison (ranked by DirAcc) ──")
    print(metrics_df.round(5).to_string())

    best_strategy = metrics_df.index[0]
    best_preds    = strategies[best_strategy]
    best_metrics  = results[best_strategy]

    print(f"\n  ✓ Best strategy: [{best_strategy}]"
          f"  DirAcc={best_metrics['DirAcc']:.4f}"
          f"  Sharpe={best_metrics['Sharpe']:+.4f}")

    return {
        "metrics_df":    metrics_df,
        "best_strategy": best_strategy,
        "best_preds":    best_preds,
        "best_metrics":  best_metrics,
    }


# ─────────────────────────────────────────────
# 5.  MAIN
# ─────────────────────────────────────────────

def main():
    print("Loading data …")
    pivot = load_and_pivot(DATA_PATH)
    print(f"  {len(pivot):,} rows | {pivot['fold_id'].nunique()} folds | "
          f"{pivot['date'].min().date()} → {pivot['date'].max().date()}")

    result_a = evaluate_group(pivot, GROUP_A, "A (LSTM + LightGBM)")
    result_b = evaluate_group(pivot, GROUP_B, "B (TCN + TFT + XGBoost)")

    # ── Overall winner ──
    print(f"\n{'='*60}")
    print(f"  OVERALL WINNER")
    print(f"{'='*60}")

    da_a = result_a["best_metrics"]["DirAcc"]
    da_b = result_b["best_metrics"]["DirAcc"]

    if da_a >= da_b:
        winner_group    = "A (LSTM + LightGBM)"
        winner_strategy = result_a["best_strategy"]
        winner_metrics  = result_a["best_metrics"]
    else:
        winner_group    = "B (TCN + TFT + XGBoost)"
        winner_strategy = result_b["best_strategy"]
        winner_metrics  = result_b["best_metrics"]

    print(f"\n  Group A best → [{result_a['best_strategy']}]  DirAcc={da_a:.4f}")
    print(f"  Group B best → [{result_b['best_strategy']}]  DirAcc={da_b:.4f}")
    print(f"\n  ► WINNER: Group {winner_group}  |  Strategy: {winner_strategy}")
    print(f"    DirAcc={winner_metrics['DirAcc']:.4f}  "
          f"Sharpe={winner_metrics['Sharpe']:+.4f}  "
          f"RMSE={winner_metrics['RMSE']:.5f}")
    print(f"{'='*60}")

    # ── Save outputs ──
    out = pivot[["date", "fold_id", "y_true"]].copy()
    out["ensemble_A"] = result_a["best_preds"]
    out["ensemble_B"] = result_b["best_preds"]
    out.to_csv("ensemble_predictions.csv", index=False)
    print("\n  Saved: ensemble_predictions.csv")

    return out, result_a, result_b


if __name__ == "__main__":
    out, result_a, result_b = main()

Loading data …
  727 rows | 35 folds | 2023-02-01 → 2025-12-23

  GROUP A (LSTM + LightGBM):  ['LSTM', 'LightGBM']

── Individual model metrics ──
  LSTM             DirAcc=0.5475  Sharpe=+1.513  RMSE=0.01928
  LightGBM         DirAcc=0.5516  Sharpe=+1.462  RMSE=0.01918

Sharpe weights: {'LSTM': 0.509, 'LightGBM': 0.491}

── Ensemble strategy comparison (ranked by DirAcc) ──
                  DirAcc   Sharpe     RMSE  MeanRet   RetStd
Strategy                                                    
simple_avg       0.56121  1.41693  0.01912  0.00170  0.01906
sharpe_weighted  0.55846  1.38697  0.01912  0.00167  0.01906

  ✓ Best strategy: [simple_avg]  DirAcc=0.5612  Sharpe=+1.4169

  GROUP B (TCN + TFT + XGBoost):  ['TCN', 'TFT', 'XGBoost']

── Individual model metrics ──
  TCN              DirAcc=0.5447  Sharpe=+1.257  RMSE=0.01916
  TFT              DirAcc=0.5227  Sharpe=+0.271  RMSE=0.01912
  XGBoost          DirAcc=0.5420  Sharpe=+0.700  RMSE=0.01917

Sharpe weights: {'TCN': 0.564, 'TF